# Lektion 2: Logische Ausführungsreihenfolge

Übungsnotebook zu Lektion 2 · Hilfsmittel: Skript Lektion 2

Umsatz einer Position = `menge * einzelpreis`. Stornierte Bestellungen zählen nicht zum Umsatz.

**Wichtig:** SQLite ist an einigen Stellen großzügiger als der SQL-Standard. Wenn eine Aufgabe nach
„standardkonform“ fragt, zählt der Standard, auch wenn SQLite die falsche Variante ausführt.

## Vorbereitung

Führen Sie zuerst diese Zelle aus (Umschalt + Enter). Danach steht Ihnen `sql(""" ... """)` für Abfragen zur Verfügung.

In [ ]:
import sqlite3
import pandas as pd

pd.set_option("display.max_rows", 100)
con = sqlite3.connect("../daten/shop_oltp.db")

def sql(abfrage):
    """Führt eine SELECT-Abfrage aus und zeigt das Ergebnis als Tabelle."""
    try:
        return pd.read_sql_query(abfrage, con)
    except Exception as e:
        print("Fehler:", str(e).split("': ")[-1])

def ausfuehren(befehl):
    """Führt Befehle ohne Ergebnistabelle aus (CREATE, INSERT, DROP ...)."""
    con.executescript(befehl)
    con.commit()
    print("ausgeführt")

---
### Aufgabe 1: Die richtige Reihenfolge

Bringen Sie die Klauseln in die **logische Ausführungsreihenfolge**:

`ORDER BY` · `WHERE` · `SELECT` · `HAVING` · `FROM` · `LIMIT` · `GROUP BY`

*Ihre Antwort:*

---
### Aufgabe 2: Zwischenergebnisse sichtbar machen

Die Abfrage unten beantwortet die Frage aus dem Skript: Welche Regionen hatten 2025 mehr als 40.000 € Umsatz?
Führen Sie sie aus.

Ermitteln Sie danach mit eigenen Abfragen:

a) Wie viele Zeilen gibt es nach FROM und JOIN?
b) Wie viele Zeilen bleiben nach WHERE übrig?
c) Wie viele Gruppen entstehen durch GROUP BY?

Tipp: Ersetzen Sie die SELECT-Liste durch `COUNT(*)` und lassen Sie die späteren Klauseln weg.

In [ ]:
sql("""
SELECT r.name AS region,
       SUM(p.menge * p.einzelpreis) AS umsatz
FROM bestellposition p
JOIN bestellung b ON b.bestellung_id = p.bestellung_id
JOIN kunde k      ON k.kunde_id      = b.kunde_id
JOIN ort o        ON o.ort_id        = k.ort_id
JOIN region r     ON r.region_id     = o.region_id
WHERE b.status <> 'storniert'
  AND b.bestelldatum BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY r.name
HAVING SUM(p.menge * p.einzelpreis) > 40000
ORDER BY umsatz DESC
""")

In [ ]:
sql("""

""")

In [ ]:
sql("""

""")

In [ ]:
sql("""

""")

*Ihre Antwort:*

---
### Aufgabe 3: Alias in WHERE

Die folgende Abfrage soll alle Bestellpositionen mit einem Betrag über 2.000 € liefern. In SQLite läuft sie.

a) Warum würde sie in PostgreSQL oder SQL Server mit einem Fehler abbrechen?
b) Schreiben Sie die Abfrage standardkonform.

In [ ]:
sql("""
SELECT bestellung_id, position_nr, menge * einzelpreis AS betrag
FROM bestellposition
WHERE betrag > 2000
""")

In [ ]:
sql("""

""")

*Ihre Antwort:*

---
### Aufgabe 4: Aggregatfunktion in WHERE

Führen Sie die Abfrage aus und lesen Sie die Fehlermeldung.
Korrigieren Sie die Abfrage anschließend und ergänzen Sie Vor- und Nachname der Kunden.

In [ ]:
sql("""
SELECT kunde_id, COUNT(*) AS anzahl
FROM bestellung
WHERE COUNT(*) > 10
GROUP BY kunde_id
""")

In [ ]:
sql("""

""")

---
### Aufgabe 5: Gleiche Frage, andere Zahlen

Beide Abfragen sollen den Umsatz je Kategorie ohne stornierte Bestellungen berechnen. Führen Sie beide aus.

a) Vergleichen Sie die Ergebnisse. Welche Abfrage ist richtig?
b) Erklären Sie, was bei der anderen Abfrage passiert.

In [ ]:
# Variante A
sql("""
SELECT k.name AS kategorie, SUM(p.menge * p.einzelpreis) AS umsatz
FROM bestellposition p
JOIN bestellung b ON b.bestellung_id = p.bestellung_id
JOIN artikel a    ON a.artikel_id    = p.artikel_id
JOIN kategorie k  ON k.kategorie_id  = a.kategorie_id
WHERE b.status <> 'storniert'
GROUP BY k.name
""")

In [ ]:
# Variante B
sql("""
SELECT k.name AS kategorie, SUM(p.menge * p.einzelpreis) AS umsatz
FROM bestellposition p
JOIN bestellung b ON b.bestellung_id = p.bestellung_id
JOIN artikel a    ON a.artikel_id    = p.artikel_id
JOIN kategorie k  ON k.kategorie_id  = a.kategorie_id
GROUP BY k.name
HAVING b.status <> 'storniert'
""")

*Ihre Antwort:*

---
### Aufgabe 6: Nicht gruppierte Spalte

Die Abfrage soll zeigen, wie oft Artikel jeder Kategorie bestellt wurden.

a) Was ist an der Abfrage falsch, obwohl SQLite sie ausführt?
b) Schreiben Sie zwei korrekte Varianten: (1) Anzahl Positionen je Kategorie, (2) Anzahl Positionen je Kategorie und Artikel.

In [ ]:
sql("""
SELECT k.name AS kategorie, a.bezeichnung, COUNT(*) AS positionen
FROM bestellposition p
JOIN artikel a   ON a.artikel_id   = p.artikel_id
JOIN kategorie k ON k.kategorie_id = a.kategorie_id
GROUP BY k.name
""")

In [ ]:
sql("""

""")

In [ ]:
sql("""

""")

*Ihre Antwort:*

---
### Aufgabe 7: Rangliste mit LIMIT und OFFSET

a) Welche **drei Artikel** haben insgesamt den höchsten Umsatz gebracht (ohne Stornos)?
b) Zeigen Sie die **Plätze 4 bis 6** dieser Rangliste.

In [ ]:
sql("""

""")

In [ ]:
sql("""

""")

---
### Aufgabe 8: DISTINCT

Das Marketing plant eine Aktion für Notebook-Zubehör. In welchen **Orten** wohnen Kunden, die **2025 das „Notebook Air 13“** gekauft haben (ohne Stornos)? Jeder Ort soll nur einmal erscheinen, alphabetisch sortiert.

Führen Sie die Abfrage auch einmal ohne DISTINCT aus und vergleichen Sie die Zeilenzahl.

In [ ]:
sql("""

""")

---
### Aufgabe 9: Index und Ausführungsplan

Mit `EXPLAIN QUERY PLAN` zeigt SQLite, wie es eine Abfrage ausführt.

1. Führen Sie die beiden Pläne aus und notieren Sie, ob SQLite **SCAN** oder **SEARCH** meldet.
2. Legen Sie mit der vorbereiteten Zelle einen Index auf `bestelldatum` an.
3. Führen Sie beide Pläne noch einmal aus. Was ändert sich und warum?
4. Löschen Sie den Index am Ende wieder (letzte Zelle).

In [ ]:
# Plan 1: Bedingung direkt auf der Spalte
sql("""
EXPLAIN QUERY PLAN
SELECT * FROM bestellung
WHERE bestelldatum BETWEEN '2025-01-01' AND '2025-12-31'
""")

In [ ]:
# Plan 2: Spalte in einer Funktion
sql("""
EXPLAIN QUERY PLAN
SELECT * FROM bestellung
WHERE strftime('%Y', bestelldatum) = '2025'
""")

In [ ]:
ausfuehren("CREATE INDEX idx_bestelldatum ON bestellung(bestelldatum)")

In [ ]:
# Index wieder entfernen
ausfuehren("DROP INDEX IF EXISTS idx_bestelldatum")

*Ihre Antwort:*

---
### Aufgabe 10: Prüfungsaufgabe: Fehler finden

Ein Kollege möchte wissen, in welchen Orten mehr als zwei Kunden wohnen, mit der Anzahl Kunden, absteigend sortiert.
Seine Abfrage:

```sql
SELECT o.name AS ort, k.nachname, COUNT(*) AS anzahl
FROM kunde k
JOIN ort o ON o.ort_id = k.ort_id
WHERE anzahl > 2
GROUP BY o.name
ORDER BY anzahl DESC
```

a) Nennen Sie alle Fehler und begründen Sie sie mit der Ausführungsreihenfolge.
b) Schreiben Sie die korrigierte Abfrage und führen Sie sie aus.

In [ ]:
sql("""

""")

*Ihre Antwort:*